# Coding Agent Datasets → Oxidize SFT JSONL

Pipeline: **stream HF coding traces → JSONL → (optional) self-train → upload adapter to HF → download & run**.

**Repo:** [Zapdev-labs/oxidize](https://github.com/Zapdev-labs/oxidize) — branch **`oxidize-c-gemma-bench`** (`self-train` not on `master` yet).

| Step | Default | Time |
|------|---------|------|
| Stream + convert JSONL | always | ~1–2 min |
| Train (`RUN_TRAIN=True`) | off | hours (CPU) |
| Upload adapter to HF | on when training finishes | ~1 min |
| Download & inference | manual / last cell | varies |

Set **`HF_TOKEN`** (Colab Secrets or env) for upload/download. Set **`HF_ADAPTER_REPO`** to your model repo (e.g. `youruser/my-coding-lora`).

In [ ]:
import json, os, sys
from pathlib import Path

# Colab clones this branch — self-train is not on master yet
GH_REPO = "https://github.com/Zapdev-labs/oxidize.git"
GH_BRANCH = "oxidize-c-gemma-bench"

# --- training / HF publish (flip RUN_TRAIN when you want the full loop) ---
RUN_TRAIN = False
UPLOAD_TO_HF = True
HF_ADAPTER_REPO = os.environ.get("HF_ADAPTER_REPO", "")  # e.g. freakyskittle/my-coding-agent-lora
HF_PRIVATE = True

# Base GGUF for training + inference (HF repo + filename, or local path)
BASE_GGUF_HF = os.environ.get("BASE_GGUF_HF", "")       # e.g. freakyskittle/gemma-4-31B-it-AL-GGUF
BASE_GGUF_FILE = os.environ.get("BASE_GGUF_FILE", "")   # e.g. gemma-4-31B-it-AL5.gguf
BASE_GGUF_LOCAL = os.environ.get("BASE_GGUF_LOCAL", "") # override: absolute local path

TRAIN_OUT = Path(os.environ.get("TRAIN_OUT", "self-train-out"))
SELF_TRAIN_ROUNDS = int(os.environ.get("SELF_TRAIN_ROUNDS", "2"))
PROMPTS_PER_ROUND = int(os.environ.get("PROMPTS_PER_ROUND", "4"))

IN_COLAB = "google.colab" in sys.modules
ROOT = Path("/content/oxidize") if IN_COLAB else Path("..").resolve()
OUT_DIR = ROOT / "data" / "sft_samples"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_ROWS = int(os.environ.get("MAX_ROWS", "5"))

!{sys.executable} -m pip install -q datasets huggingface_hub

from datasets import load_dataset

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
print("repo:", GH_REPO)
print("branch:", GH_BRANCH)
print("RUN_TRAIN:", RUN_TRAIN, "| UPLOAD_TO_HF:", UPLOAD_TO_HF)
print("HF_ADAPTER_REPO:", HF_ADAPTER_REPO or "(set HF_ADAPTER_REPO)")
print("root:", ROOT)

## Dataset catalog (no download)

Curated coding / agent-trace sources similar to Fable, GLM-5.2-Agent, and SWE trajectories.

In [ ]:
CATALOG = [
    {"id": "Glint-Research/Fable-5-traces", "kind": "fable", "split": "train", "note": "canonical Fable hub"},
    {"id": "Nexlab/fable5-agentic-coding-sft", "kind": "fable", "split": "train", "note": "~160k curated tool-loop SFT"},
    {"id": "AletheiaResearch/GLM-5.2-Agent", "kind": "glm", "split": "train", "note": "319 teich GLM agent sessions"},
    {"id": "TIGER-Lab/SWE-Next-SFT-Trajectories", "kind": "swe", "split": "train", "note": "3.7k ShareGPT SWE trajectories"},
    {"id": "SWE-Lego/SWE-Lego-Real-Data", "kind": "swe", "split": "resolved", "note": "real GitHub issue trajectories"},
    {"id": "nvidia/Open-SWE-Traces", "kind": "swe", "split": "train", "note": "207k OpenHands/SWE-agent traces"},
    {"id": "trace-commons/agent-traces", "kind": "raw", "split": "train", "note": "donated Claude/Codex/Cursor sessions"},
    {"id": "RESMP-DEV/agent-traces-curated-2026", "kind": "mix", "split": "train", "note": "1.1M mixed agent traces"},
]

for i, d in enumerate(CATALOG, 1):
    print(f"{i:2}. {d['id']:<45} [{d['kind']}] {d['note']}")

## Stream a handful of rows

Tries small/fast datasets first. Stops after `MAX_ROWS` converted examples.

In [ ]:
def row_to_messages(row: dict) -> list[dict] | None:
    """Normalize heterogeneous HF rows → OpenAI-style messages."""
    if isinstance(row.get("messages"), list) and row["messages"]:
        return row["messages"]
    if isinstance(row.get("trajectory"), list) and row["trajectory"]:
        return row["trajectory"]
    if isinstance(row.get("conversations"), list) and row["conversations"]:
        return row["conversations"]
    # Crownelius-style wrapped JSON
    rj = row.get("row_json")
    if isinstance(rj, str):
        try:
            inner = json.loads(rj)
            if isinstance(inner.get("messages"), list):
                return inner["messages"]
        except json.JSONDecodeError:
            pass
    # plain text fallbacks
    for key in ("text", "content", "prompt"):
        if isinstance(row.get(key), str) and row[key].strip():
            return [{"role": "user", "content": row[key]}]
    return None


def messages_to_chat_text(messages: list[dict]) -> str:
    IM_END = "<|" + "im_end" + "|>"
    parts: list[str] = []
    for m in messages:
        role = str(m.get("role", "user"))
        content = m.get("content")
        if content is None:
            content = m.get("text", "")
        if not isinstance(content, str):
            content = json.dumps(content, ensure_ascii=False)
        if role == "tool":
            # oxidize dataset loader ignores tool role today — fold into user block
            parts.append(f"<|im_start|>user\n[tool] {content}\n{IM_END}\n")
            continue
        parts.append(f"<|im_start|>{role}\n{content}\n{IM_END}\n")
    return "".join(parts)


def to_oxidize_row(messages: list[dict]) -> dict:
    # oxidize-finetuning accepts {"messages": ...} or {"text": ...}
    return {"messages": messages}


STREAM_ORDER = [
    ("TIGER-Lab/SWE-Next-SFT-Trajectories", "train"),
    ("AletheiaResearch/GLM-5.2-Agent", "train"),
    ("Nexlab/fable5-agentic-coding-sft", "train"),
    ("Glint-Research/Fable-5-traces", "train"),
]

converted: list[dict] = []
source_used: str | None = None

for repo_id, split in STREAM_ORDER:
    if len(converted) >= MAX_ROWS:
        break
    try:
        ds = load_dataset(repo_id, split=split, streaming=True, token=token)
        for row in ds:
            msgs = row_to_messages(row)
            if not msgs:
                continue
            converted.append(to_oxidize_row(msgs))
            source_used = repo_id
            if len(converted) >= MAX_ROWS:
                break
        if converted:
            break
    except Exception as e:
        print(f"skip {repo_id}: {e}")

if not converted:
    # offline fallback — still produces valid oxidize JSONL
    converted = [
        {"messages": [
            {"role": "user", "content": "Write a Python function to reverse a string."},
            {"role": "assistant", "content": "def rev(s: str) -> str:\n    return s[::-1]"},
        ]},
        {"messages": [
            {"role": "user", "content": "Explain what a KV cache does in LLM inference."},
            {"role": "assistant", "content": "It stores key/value projections for prior tokens so decode steps avoid recomputing attention over the full prefix."},
        ]},
    ]
    source_used = "synthetic-fallback"

print(f"source: {source_used}")
print(f"rows:   {len(converted)}")

In [ ]:
sample_path = OUT_DIR / "coding_agent_sample.jsonl"
with sample_path.open("w", encoding="utf-8") as f:
    for row in converted:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("wrote", sample_path, f"({sample_path.stat().st_size} bytes)")
print("--- first row preview ---")
print(json.dumps(converted[0], ensure_ascii=False, indent=2)[:1200])

## Build oxidize-finetuning (quick smoke test)

Always runs. Full training + HF upload is in the next section (`RUN_TRAIN=True`).

In [ ]:
import subprocess

if IN_COLAB:
    clone_cmd = f"git clone -q --depth 1 -b {GH_BRANCH} {GH_REPO} /content/oxidize"
    get_ipython().system(clone_cmd)
    ROOT = Path("/content/oxidize")
    OUT_DIR = ROOT / "data" / "sft_samples"
    OUT_DIR.mkdir(parents=True, exist_ok=True)

r = subprocess.run(
    ["cargo", "build", "-p", "oxidize-finetuning", "-q"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
if r.returncode != 0:
    print(r.stderr or r.stdout)
    print(f"\nHint: self-train requires branch {GH_BRANCH} — master does not have it yet.")
else:
    print("built oxidize-finetuning OK")

sample = OUT_DIR / "coding_agent_sample.jsonl"
print("\n# one-shot SFT (needs a real GGUF)")
print(f"cargo run -p oxidize-finetuning -- sft \\")
print(f"  --model /path/to/model.gguf \\")
print(f"  --dataset {sample} \\")
print(f"  --output lora-out --epochs 1 --max-tokens 2048")
print("\n# iterative self-train (checkpoints + self-dialogue)")
print(f"cargo run -p oxidize-finetuning -- self-train \\")
print(f"  --model /path/to/model.gguf \\")
print(f"  --dataset {sample} \\")
print(f"  --output self-train-out --rounds 2 --prompts-per-round 4 --epochs-per-round 1")

## Train + upload adapter to Hugging Face

Set `RUN_TRAIN = True` in cell 1. Requires `HF_TOKEN`, `HF_ADAPTER_REPO`, and a base GGUF (`BASE_GGUF_HF` + `BASE_GGUF_FILE` or `BASE_GGUF_LOCAL`).

Uploads `adapter/` (+ `synthetic.jsonl`, metrics) to your HF **model** repo when training finishes.

In [ ]:
import subprocess
from huggingface_hub import HfApi, create_repo, hf_hub_download, upload_folder

adapter_dir = None
base_gguf = None

if RUN_TRAIN:
    if IN_COLAB:
        get_ipython().system(f"git clone -q --depth 1 -b {GH_BRANCH} {GH_REPO} /content/oxidize")
        ROOT = Path("/content/oxidize")

    if BASE_GGUF_LOCAL:
        base_gguf = Path(BASE_GGUF_LOCAL)
    elif BASE_GGUF_HF and BASE_GGUF_FILE:
        base_gguf = Path(hf_hub_download(repo_id=BASE_GGUF_HF, filename=BASE_GGUF_FILE, token=token))
    else:
        raise ValueError("Set BASE_GGUF_LOCAL or BASE_GGUF_HF + BASE_GGUF_FILE")

    train_out = (ROOT / TRAIN_OUT).resolve()
    sample_path = OUT_DIR / "coding_agent_sample.jsonl"
    subprocess.run(["cargo", "build", "-p", "oxidize-finetuning", "--release", "-q"], cwd=ROOT, check=True)

    cmd = [
        "cargo", "run", "-p", "oxidize-finetuning", "--release", "--",
        "self-train", "--model", str(base_gguf), "--dataset", str(sample_path),
        "--output", str(train_out), "--rounds", str(SELF_TRAIN_ROUNDS),
        "--prompts-per-round", str(PROMPTS_PER_ROUND), "--epochs-per-round", "1",
        "--max-tokens", "2048",
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)
    adapter_dir = train_out / "adapter"
    print("done:", adapter_dir)
else:
    print("RUN_TRAIN=False — skipped")

if UPLOAD_TO_HF and adapter_dir and adapter_dir.is_dir():
    if not HF_ADAPTER_REPO:
        raise ValueError("Set HF_ADAPTER_REPO")
    if not token:
        raise ValueError("Set HF_TOKEN")
    api = HfApi(token=token)
    create_repo(HF_ADAPTER_REPO, repo_type="model", private=HF_PRIVATE, exist_ok=True, token=token)
    if HF_PRIVATE:
        api.update_repo_settings(HF_ADAPTER_REPO, private=True, repo_type="model", token=token)
    upload_folder(folder_path=str(adapter_dir), repo_id=HF_ADAPTER_REPO, repo_type="model",
                  commit_message="oxidize self-train adapter", token=token)
    for extra in ("synthetic.jsonl", "metrics.csv", "self_train_state.json"):
        p = (ROOT / TRAIN_OUT / extra)
        if p.is_file():
            api.upload_file(str(p), path_in_repo=extra, repo_id=HF_ADAPTER_REPO, repo_type="model", token=token)
    print(f"https://huggingface.co/{HF_ADAPTER_REPO}")
elif UPLOAD_TO_HF:
    print("nothing to upload — set RUN_TRAIN=True first")

## Use the model after upload

Training produces a **LoRA adapter** (not a standalone GGUF). At inference you need:
1. The **same base GGUF** you trained on
2. The **adapter** from HF (`adapter_manifest.json`)

Resume training or continue self-train from the uploaded checkpoint.

In [ ]:
from huggingface_hub import snapshot_download

if not HF_ADAPTER_REPO:
    print("Set HF_ADAPTER_REPO to download your adapter")
else:
    local_adapter = Path("/content/adapter") if IN_COLAB else (ROOT / "hf_adapter")
    snapshot_download(repo_id=HF_ADAPTER_REPO, local_dir=str(local_adapter), token=token)
    print("downloaded:", local_adapter)

    if BASE_GGUF_LOCAL:
        gguf = BASE_GGUF_LOCAL
    elif BASE_GGUF_HF and BASE_GGUF_FILE:
        gguf = hf_hub_download(repo_id=BASE_GGUF_HF, filename=BASE_GGUF_FILE, token=token)
    else:
        gguf = "/path/to/base.gguf"

    print("\n# continue self-train from HF adapter")
    print(f"cargo run -p oxidize-finetuning -- self-train \\")
    print(f"  --model {gguf} \\")
    print(f"  --dataset {OUT_DIR / 'coding_agent_sample.jsonl'} \\")
    print(f"  --resume-from {local_adapter} \\")
    print(f"  --output self-train-out")
    print("\n# chat with oxidize (base model; LoRA GGUF merge coming soon)")
    print(f"cargo run -p oxidize-cli -- --model {gguf} --prompt 'Hello'")

## Quality notes

- Prefer **content-verified** Fable rows (`toolu_…` IDs) over keyword-matched "mythos" dumps.
- [Crownelius/Complete-FABLE.5-traces-2M](https://huggingface.co/datasets/Crownelius/Complete-FABLE.5-traces-2M) is ~50k verified rows after cleanup, not 2M.
- For SWE agents: `nvidia/Open-SWE-Traces`, `SWE-Lego/*`, `TIGER-Lab/SWE-Next-SFT-Trajectories`.
- Scale up by raising `MAX_ROWS` and writing separate JSONL shards per source — still use **streaming** to avoid multi-GB downloads in-notebook.